In [1]:
import pandas as pd
url = "https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset/resolve/main/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv"
df = pd.read_csv(url)
df.head()

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [2]:
# Check dataset shape
print("dataset shape:", df.shape)

dataset shape: (26872, 5)


In [3]:
# Check columns
df.columns

Index(['flags', 'instruction', 'category', 'intent', 'response'], dtype='object')

In [4]:
# Display dataset information including columns and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26872 entries, 0 to 26871
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   flags        26872 non-null  object
 1   instruction  26872 non-null  object
 2   category     26872 non-null  object
 3   intent       26872 non-null  object
 4   response     26872 non-null  object
dtypes: object(5)
memory usage: 1.0+ MB


In [5]:
# Check missing values
df.isnull().sum()

,0
flags,0
instruction,0
category,0
intent,0
response,0


In [6]:
# Check for completely duplicate rows in dataset
print("duplicate rows:", df.duplicated().sum())


duplicate rows: 0


In [7]:
df["intent"].value_counts()

,count
intent,
contact_customer_service,1000
complaint,1000
check_invoice,1000
switch_account,1000
edit_account,1000
contact_human_agent,999
check_payment_methods,999
delivery_period,999
newsletter_subscription,999


In [8]:
df["instruction"].head(10)

,instruction
0,question about cancelling order {{Order Number}}
1,i have a question about cancelling oorder {{Or...
2,i need help cancelling puchase {{Order Number}}
3,I need to cancel purchase {{Order Number}}
4,"I cannot afford this order, cancel purchase {{..."
5,can you help me cancel order {{Order Number}}?
6,"I can no longer afford order {{Order Number}},..."
7,I am trying to cancel purchase {{Order Number}}
8,I have got to cancel purchase {{Order Number}}
9,i need help canceling purchase {{Order Number}}


In [9]:
df["response"].head(10)

,response
0,I've understood you have a question regarding ...
1,I've been informed that you have a question ab...
2,I can sense that you're seeking assistance wit...
3,I understood that you need assistance with can...
4,I'm sensitive to the fact that you're facing f...
5,"Of course, I'm here to assist you in canceling..."
6,I pick up what you're putting down that you're...
7,I've understood that you're seeking assistance...
8,I'm sensitive to the fact that you're seeking ...
9,I perceive that you're seeking assistance with...


In [10]:
#Check for duplicate instructions
print("duplicate instructions:", df["instruction"].duplicated().sum())


duplicate instructions: 2237


In [11]:
#Check for duplicate response
print("duplicate response:", df["response"].duplicated().sum())

duplicate response: 2


In [12]:
#Check the placeholders in instructions
import re
df["instruction"].str.contains(r"\{\{.*?\}\}", regex=True).sum()

np.int64(6670)

In [13]:
#Check the placeholders in responses
df["response"].str.contains(r"\{\{.*?\}\}", regex=True).sum()

np.int64(13006)

In [14]:
#Check duplicate instructions and their intents
duplicate_instructions = df[df["instruction"].duplicated(keep=False)]
duplicate_instructions[["instruction", "intent", "response"]].head(20)

,instruction,intent,response
0,question about cancelling order {{Order Number}},cancel_order,I've understood you have a question regarding ...
2,i need help cancelling puchase {{Order Number}},cancel_order,I can sense that you're seeking assistance wit...
6,"I can no longer afford order {{Order Number}},...",cancel_order,I pick up what you're putting down that you're...
7,I am trying to cancel purchase {{Order Number}},cancel_order,I've understood that you're seeking assistance...
8,I have got to cancel purchase {{Order Number}},cancel_order,I'm sensitive to the fact that you're seeking ...
9,i need help canceling purchase {{Order Number}},cancel_order,I perceive that you're seeking assistance with...
10,i dont know what to do to cancel order {{Order...,cancel_order,I've taken note that you're unsure of the canc...
11,I have a problem with cancelling purchase {{Or...,cancel_order,I'm following you that you're facing difficult...
12,problem with canceling purchase {{Order Number}},cancel_order,I'm aware that you're facing difficulties canc...
13,can you help me canceling order {{Order Number}}?,cancel_order,I comprehend the urgency and complexity involv...


In [15]:
duplicate_instructions.groupby("instruction")["intent"].nunique().value_counts()

,count
intent,
1,989


In [16]:
#Create clean_instruction
df["clean_instruction"] = df["instruction"].apply(
    lambda x: re.sub(r"\{\{.*?\}\}", "", x)
)

In [17]:
# Remove extra spaces
df["clean_instruction"] = df["clean_instruction"].str.replace(
    r"\s+", " ", regex=True
).str.strip()

In [18]:
# Convert to lowercase
df["clean_instruction"] = df["clean_instruction"].str.lower()

In [19]:
# Check original vs cleaned text
df[["instruction", "clean_instruction", "intent", "response"]].head(10)

,instruction,clean_instruction,intent,response
0,question about cancelling order {{Order Number}},question about cancelling order,cancel_order,I've understood you have a question regarding ...
1,i have a question about cancelling oorder {{Or...,i have a question about cancelling oorder,cancel_order,I've been informed that you have a question ab...
2,i need help cancelling puchase {{Order Number}},i need help cancelling puchase,cancel_order,I can sense that you're seeking assistance wit...
3,I need to cancel purchase {{Order Number}},i need to cancel purchase,cancel_order,I understood that you need assistance with can...
4,"I cannot afford this order, cancel purchase {{...","i cannot afford this order, cancel purchase",cancel_order,I'm sensitive to the fact that you're facing f...
5,can you help me cancel order {{Order Number}}?,can you help me cancel order ?,cancel_order,"Of course, I'm here to assist you in canceling..."
6,"I can no longer afford order {{Order Number}},...","i can no longer afford order , cancel it",cancel_order,I pick up what you're putting down that you're...
7,I am trying to cancel purchase {{Order Number}},i am trying to cancel purchase,cancel_order,I've understood that you're seeking assistance...
8,I have got to cancel purchase {{Order Number}},i have got to cancel purchase,cancel_order,I'm sensitive to the fact that you're seeking ...
9,i need help canceling purchase {{Order Number}},i need help canceling purchase,cancel_order,I perceive that you're seeking assistance with...


In [20]:
print(df["category"].value_counts())

category
ACCOUNT         5986
ORDER           3988
REFUND          2992
CONTACT         1999
INVOICE         1999
PAYMENT         1998
FEEDBACK        1997
DELIVERY        1994
SHIPPING        1970
SUBSCRIPTION     999
CANCEL           950
Name: count, dtype: int64


In [21]:
unique_df = df[["instruction", "intent"]].drop_duplicates(
    subset=["instruction"]
)

print("Unique instructions:", unique_df.shape)

Unique instructions: (24635, 2)


In [22]:
# Split into 80% training and 20% testing
from sklearn.model_selection import train_test_split

train_instructions, test_instructions = train_test_split(
    unique_df,
    test_size=0.20,
    random_state=42,
    stratify=unique_df["intent"]
)

In [23]:
# Create the actual training and testing datasets
train_df = df[
    df["instruction"].isin(train_instructions["instruction"])
].copy()

test_df = df[
    df["instruction"].isin(test_instructions["instruction"])
].copy()

In [24]:
print("Training data:", train_df.shape)
print("Testing data:", test_df.shape)

Training data: (21556, 6)
Testing data: (5316, 6)


In [25]:
# Check for data leakage
common = set(train_df["instruction"]) & set(test_df["instruction"])
print("Instructions appearing in both:", len(common))

Instructions appearing in both: 0


**STEP 4 — Create Semantic Embeddings**

In [26]:
!pip install -q sentence-transformers

In [27]:
from sentence_transformers import SentenceTransformer

In [28]:
# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [29]:
# Create embeddings for training questions
train_embeddings = model.encode(
    train_df["clean_instruction"].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/674 [00:00<?, ?it/s]

In [30]:
print(train_embeddings.shape)

(21556, 384)


**STEP 5 — Semantic Search using Cosine Similarity**

In [31]:
# Import cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

In [32]:
query = test_df.iloc[0]["clean_instruction"]

print("Customer question:", query)
print("Actual intent:", test_df.iloc[0]["intent"])

Customer question: i have a question about cancelling oorder
Actual intent: cancel_order


In [33]:
# Convert query into embedding
query_embedding = model.encode([query])

print(query_embedding.shape)

(1, 384)


In [34]:
# Calculate similarity
similarities = cosine_similarity(
    query_embedding,
    train_embeddings
)[0]

In [35]:
query_embedding = model.encode([query])
similarities = cosine_similarity(
    query_embedding,
    train_embeddings
)[0]

# Find most similar question
best_index = similarities.argmax()

print("Customer question:")
print(query)

print("\nMost similar training question:")
print(train_df.iloc[best_index]["instruction"])

print("\nSimilarity score:")
print(similarities[best_index])

print("\nPredicted intent:")
print(train_df.iloc[best_index]["intent"])

Customer question:
i have a question about cancelling oorder

Most similar training question:
I have an question about cancelling order {{Order Number}}

Similarity score:
0.8279273

Predicted intent:
cancel_order


In [36]:
print("\nChatbot response:")
print(train_df.iloc[best_index]["response"])


Chatbot response:
I certainly recognize that you have a question regarding canceling your order with the order number {{Order Number}}. I'm here to assist you with any concerns or queries you may have. Please go ahead and let me know what specific question or doubt you have, and I'll provide you with the necessary information and guidance.


**STEP 8 — Make the Semantic Search Reusable**

In [37]:
# Create the chatbot search function
def semantic_search(query):

    # Clean the customer question
    clean_query = re.sub(r"\{\{.*?\}\}", "", query)
    clean_query = re.sub(r"\s+", " ", clean_query).strip().lower()

    # Convert question into embedding
    query_embedding = model.encode([clean_query])

    # Calculate similarity with training questions
    similarities = cosine_similarity(
        query_embedding,
        train_embeddings
    )[0]

    # Find the most similar question
    best_index = similarities.argmax()

    # Get information from the matching row
    matched_question = train_df.iloc[best_index]["instruction"]
    intent = train_df.iloc[best_index]["intent"]
    response = train_df.iloc[best_index]["response"]
    score = similarities[best_index]

    return matched_question, intent, score, response

In [38]:
query = "I have a problem with my payment"
matched_question, intent, score, response = semantic_search(query)

print("Customer:", query)
print("\nMatched question:", matched_question)
print("\nIntent:", intent)
print("\nSimilarity score:", score)
print("\nChatbot response:", response)

Customer: I have a problem with my payment

Matched question: issue with payment

Intent: payment_issue

Similarity score: 0.8983784

Chatbot response: I'm sorry to hear that you're experiencing an issue with your payment. Rest assured, I'm here to assist you in resolving this matter. To better understand the problem, could you please provide more details about the specific issue you're encountering? Whether it's a declined payment, an error message, or any other issue, your insights will help us pinpoint the problem and work towards a solution. Thank you for reaching out, and I appreciate your cooperation in getting this resolved swiftly.


**STEP 9 — Test the chatbot with different questions**

In [39]:
test_questions = [
    "Where is my order 12345?",
    "I want to get my money back",
    "I have a problem with my payment",
    "How can I change my shipping address?",
    "I forgot my password"
]

for query in test_questions:

    matched_question, intent, score, response = semantic_search(query)

    print("=" * 70)
    print("Customer:", query)
    print("Matched question:", matched_question)
    print("Intent:", intent)
    print("Similarity:", score)
    print("Response:", response)


Customer: Where is my order 12345?
Matched question: I want help locating order {{Order Number}}
Intent: track_order
Similarity: 0.76132846
Response: Honored to assist! I'm clued in that you need assistance with locating the order with order number {{Order Number}}. I'm here to help you with that. To locate your order, you can check the '{{Online Order Interaction}}' section on our website. If you have any further questions or need additional support, please feel free to let me know.
Customer: I want to get my money back
Matched question: I would like to obtain my money back
Intent: get_refund
Similarity: 0.91821367
Response: I'm with you, your desire to obtain a refund for your money. We value your satisfaction as our customer and we strive to resolve any issues promptly. To start the process, I kindly request you to provide some details regarding your purchase, such as the order number, invoice number, or any other relevant information. With this information, we can locate your trans

**STEP 10 — Evaluate the Semantic Search**

In [40]:
test_embeddings = model.encode(
    test_df["clean_instruction"].tolist(),
    show_progress_bar=True
)

print(test_embeddings.shape)

Batches:   0%|          | 0/167 [00:00<?, ?it/s]

(5316, 384)


In [41]:
# Calculate similarity for all test questions
similarities = cosine_similarity(
    test_embeddings,
    train_embeddings
)
print(similarities.shape)

(5316, 21556)


In [42]:
# Predict intents
best_indices = similarities.argmax(axis=1)

predicted_intents = [
    train_df.iloc[i]["intent"]
    for i in best_indices
]

test_df["predicted_intent"] = predicted_intents

In [43]:
test_df["similarity_score"] = similarities.max(axis=1)

In [44]:
# Calculate accuracy
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(
    test_df["intent"],
    test_df["predicted_intent"]
)

print("Accuracy:", accuracy)

Accuracy: 0.989089541008277


In [45]:
# Classification report
print(classification_report(
    test_df["intent"],
    test_df["predicted_intent"]
))

                          precision    recall  f1-score   support

            cancel_order       0.98      0.99      0.99       185
            change_order       0.99      0.98      0.98       206
 change_shipping_address       0.99      0.99      0.99       195
  check_cancellation_fee       1.00      1.00      1.00       190
           check_invoice       0.96      0.95      0.95       202
   check_payment_methods       1.00      1.00      1.00       200
     check_refund_policy       0.99      0.99      0.99       199
               complaint       1.00      1.00      1.00       200
contact_customer_service       1.00      1.00      1.00       200
     contact_human_agent       0.99      0.99      0.99       200
          create_account       0.98      0.96      0.97       198
          delete_account       0.99      0.99      0.99       194
        delivery_options       0.99      1.00      1.00       170
         delivery_period       0.99      0.99      0.99       200
         

In [46]:
#Parameter / Entity Extraction step by step
#The purpose is simple:
#The semantic search tells us what the customer wants, while parameter extraction tells us the specific information mentioned by the customer.

import re

def extract_order_number(query):
    pattern = r'\b\d{4,}\b'
    match = re.search(pattern, query)

    if match:
        return match.group()
    else:
        return None

In [47]:
queries = [
    "Where is my order 12345?",
    "Can you track order 987654?",
    "I want to know where my package is",
    "My order number is 45678"
]

for q in queries:
    print(q)
    print("Order Number:", extract_order_number(q))
    print()

Where is my order 12345?
Order Number: 12345

Can you track order 987654?
Order Number: 987654

I want to know where my package is
Order Number: None

My order number is 45678
Order Number: 45678



In [48]:
def fill_parameters(response, order_number):
    if order_number:
        response = response.replace("{{Order Number}}", order_number)

    return response

In [49]:
def chatbot_search(query, threshold=0.70):
    matched_question, intent, score, response = semantic_search(query)

    if score < threshold:
        return None, None, score, None, "Sorry, I could not understand your question. Please rephrase it."

    order_number = extract_order_number(query)

    response = fill_parameters(response, order_number)

    return matched_question, intent, score, order_number, response


In [50]:
query = "Where is my order 12345?"

matched_question, intent, score, order_number, response = chatbot_search(query)

print("Intent:", intent)
print("Similarity:", score)
print("Order Number:", order_number)
print("Response:", response)

Intent: track_order
Similarity: 0.76132846
Order Number: 12345
Response: Honored to assist! I'm clued in that you need assistance with locating the order with order number 12345. I'm here to help you with that. To locate your order, you can check the '{{Online Order Interaction}}' section on our website. If you have any further questions or need additional support, please feel free to let me know.


In [51]:
print(test_df["similarity_score"].describe())

count    5316.000000
mean        0.943678
std         0.067810
min         0.457473
25%         0.930965
50%         0.965427
75%         0.984934
max         1.000000
Name: similarity_score, dtype: float64


In [52]:
thresholds = [0.70, 0.75, 0.80, 0.85, 0.90]

for threshold in thresholds:
    rejected = (test_df["similarity_score"] < threshold).sum()

    print("Threshold:", threshold)
    print("Questions rejected:", rejected)
    print("Questions accepted:", len(test_df) - rejected)
    print()

Threshold: 0.7
Questions rejected: 81
Questions accepted: 5235

Threshold: 0.75
Questions rejected: 136
Questions accepted: 5180

Threshold: 0.8
Questions rejected: 248
Questions accepted: 5068

Threshold: 0.85
Questions rejected: 463
Questions accepted: 4853

Threshold: 0.9
Questions rejected: 841
Questions accepted: 4475



In [53]:
test_queries = [
    "Where is my order 12345?",
    "I want to get my money back",
    "I have a problem with my payment",
    "How can I change my shipping address?",
    "I forgot my password",
    "What is the weather today?"
]

for query in test_queries:
    print("=" * 70)
    print("Customer:", query)

    matched_question, intent, score, order_number, response = chatbot_search(query)

    print("Intent:", intent)
    print("Similarity:", score)
    print("Order Number:", order_number)
    print("Response:", response)

Customer: Where is my order 12345?
Intent: track_order
Similarity: 0.76132846
Order Number: 12345
Response: Honored to assist! I'm clued in that you need assistance with locating the order with order number 12345. I'm here to help you with that. To locate your order, you can check the '{{Online Order Interaction}}' section on our website. If you have any further questions or need additional support, please feel free to let me know.
Customer: I want to get my money back
Intent: get_refund
Similarity: 0.91821367
Order Number: None
Response: I'm with you, your desire to obtain a refund for your money. We value your satisfaction as our customer and we strive to resolve any issues promptly. To start the process, I kindly request you to provide some details regarding your purchase, such as the order number, invoice number, or any other relevant information. With this information, we can locate your transaction and initiate the refund procedure for you. Rest assured that I'm here to guide you

In [54]:
def chatbot(query):
    matched_question, intent, score, order_number, response = chatbot_search(query)

    #if intent is None:
     #   return response

    return response

In [55]:
print(chatbot("Where is my order 12345?"))

Honored to assist! I'm clued in that you need assistance with locating the order with order number 12345. I'm here to help you with that. To locate your order, you can check the '{{Online Order Interaction}}' section on our website. If you have any further questions or need additional support, please feel free to let me know.


In [56]:
test_questions = [
    "Where is my order 12345?",
    "I want to get a refund",
    "I forgot my password",
    "My payment is not working",
    "I want to change my shipping address",
    "Can I cancel my order?",
    "What is the weather today?"
]

for question in test_questions:
    print("Customer:", question)
    print("Bot:", chatbot(question))
    print("-" * 80)

Customer: Where is my order 12345?
Bot: Honored to assist! I'm clued in that you need assistance with locating the order with order number 12345. I'm here to help you with that. To locate your order, you can check the '{{Online Order Interaction}}' section on our website. If you have any further questions or need additional support, please feel free to let me know.
--------------------------------------------------------------------------------
Customer: I want to get a refund
Bot: I can sense that you require assistance regarding how to get a refund. Rest assured, I'm here to guide you through the process and ensure a smooth resolution. To help you more effectively, could you please provide me with some additional details about your refund request? This will enable me to provide you with tailored advice and the necessary steps to initiate the refund process. Your satisfaction is our top priority, and I'm committed to ensuring that you receive the support you need.
--------------------